In [1]:
import os

In [2]:
os.chdir("../")

#### Update Entity

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    ALL_REQUIRED_FILES: list

#### Update Configuration Manager

In [4]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [5]:
class ConfigurationManager:
    def __init__(
        self,
        config_file_path: Path = CONFIG_FILE_PATH,
        params_file_path: Path = PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
    
    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            ALL_REQUIRED_FILES=config.ALL_REQUIRED_FILES
        )

        return data_validation_config

#### Update Components

In [6]:
import os
from textSummarizer.logging import logger

In [7]:
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_files(self) -> bool:
        try:
            validation_status = True
            all_files = os.listdir(os.path.join("artifacts", "data_ingestion", "samsum_dataset"))
            print(all_files)

            for file in self.config.ALL_REQUIRED_FILES:
                
                logger.info(f"Checking for file: {file}")
                if file not in all_files:
                    validation_status = False
                    with open(self.config.STATUS_FILE, "w") as f:
                        f.write(f"File: {file} is not present.")
                else:
                    with open(self.config.STATUS_FILE, "w") as f:
                        f.write(f"File: {file} is present.")
            return validation_status
        except Exception as e:
            return e

#### Update Pipeline

In [8]:
try:
    config = ConfigurationManager()
    validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=validation_config)
    data_validation.validate_all_files()
except Exception as e:
    raise e

['dataset_dict.json', 'test', 'train', 'validation']
[2026-02-25 10:10:08,213: INFO: 3522455410]: Checking for file: train
[2026-02-25 10:10:08,214: INFO: 3522455410]: Checking for file: test
[2026-02-25 10:10:08,215: INFO: 3522455410]: Checking for file: validation
